In [84]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score

In [85]:
columns = [
    "duration","protocol_type","service","flag","src_bytes","dst_bytes","land",
    "wrong_fragment","urgent","hot","num_failed_logins","logged_in","num_compromised",
    "root_shell","su_attempted","num_root","num_file_creations","num_shells",
    "num_access_files","num_outbound_cmds","is_host_login","is_guest_login","count",
    "srv_count","serror_rate","srv_serror_rate","rerror_rate","srv_rerror_rate",
    "same_srv_rate","diff_srv_rate","srv_diff_host_rate","dst_host_count",
    "dst_host_srv_count","dst_host_same_srv_rate","dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate","dst_host_srv_diff_host_rate","dst_host_serror_rate",
    "dst_host_srv_serror_rate","dst_host_rerror_rate","dst_host_srv_rerror_rate",
    "label","difficulty"
]

In [86]:
df_train = pd.read_csv("../data/raw/KDDTrain+.txt", header=None, names=columns)
df_test = pd.read_csv("../data/raw/KDDTest+.txt", header=None, names=columns)
print(df_train.shape, df_test.shape)

(125973, 43) (22544, 43)


In [87]:
x_train = df_train.drop(["label", "difficulty"], axis = 1)
y_train = df_train["label"]
y_train_binary = (y_train != "normal").astype(int)
x_test = df_test.drop(["label", "difficulty"], axis = 1)
y_test = df_test["label"]
y_test_binary = (y_test != "normal").astype(int)

In [88]:
x_train_encoded = pd.get_dummies(x_train)
x_test_encoded = pd.get_dummies(x_test)
x_test_encoded = x_test_encoded.reindex(columns=x_train_encoded.columns, fill_value=0)
print(x_train_encoded.shape)
print(x_test_encoded.shape)


(125973, 122)
(22544, 122)


In [89]:
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train_encoded)
x_test_scaled = scaler.transform(x_test_encoded)
print(x_train_scaled.shape)
print(x_test_scaled.shape)

(125973, 122)
(22544, 122)


In [90]:
np.random.seed(42)
label_fraction = 0.01
n_samples = x_train_scaled.shape[0]
n_labeled = int(n_samples * label_fraction)
labeled_indices = np.random.choice(n_samples, size=n_labeled, replace=False)
is_labeled = np.zeros(n_samples, dtype=bool)
is_labeled[labeled_indices] = True

print(f"Total samples: {n_samples}")
print(f"Labeled samples: {is_labeled.sum()} ({is_labeled.sum()/n_samples:.1%})")
print(f"Unlabeled samples: {(~is_labeled).sum()} ({(~is_labeled).sum()/n_samples:.1%})")

Total samples: 125973
Labeled samples: 1259 (1.0%)
Unlabeled samples: 124714 (99.0%)


In [91]:
x_labeled = x_train_scaled[is_labeled]
y_labeled = y_train_binary[is_labeled]
y_unlabeled_true = y_train_binary[~is_labeled]
x_unlabeled = x_train_scaled[~is_labeled]

print(x_labeled.shape, y_labeled.shape)
print(x_unlabeled.shape, y_unlabeled_true.shape)

(1259, 122) (1259,)
(124714, 122) (124714,)


In [92]:
clf = LogisticRegression(max_iter = 1000, random_state=42)
clf.fit(x_labeled, y_labeled)

probs = clf.predict_proba(x_unlabeled)
attack_prob = probs[:, 1]
print(attack_prob[:10])

[0.01365949 0.32500333 0.99999876 0.01447396 0.01346083 0.98774048
 0.99999947 0.99999855 0.99999737 0.99999909]


/Users/sayoowo5/CS4360-Senior-Experience-Project/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/sayoowo5/CS4360-Senior-Experience-Project/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/sayoowo5/CS4360-Senior-Experience-Project/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/sayoowo5/CS4360-Senior-Experience-Project/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/sayoowo5/CS4360-Senior-Experience-Project/.venv/lib/python3.9/site-packages/sklearn/line

In [93]:
high_threshold = .95
low_threshold = .05 
confident_attack = attack_prob >= high_threshold
confident_normal = attack_prob <= low_threshold
confident_mask = confident_attack | confident_normal

print(f"Confident predictions: {confident_mask.sum()} out of {len(attack_prob)} ({confident_mask.sum()/len(attack_prob):.1%})")
print(f"  - confident attack: {confident_attack.sum()}")
print(f"  - confident normal: {confident_normal.sum()}")

Confident predictions: 109000 out of 124714 (87.4%)
  - confident attack: 50907
  - confident normal: 58093


In [94]:
pseudo_labeles = np.where(confident_attack, 1,0)
pseudo_labels_confident = pseudo_labeles[confident_mask]
true_labels_confident = y_unlabeled_true[confident_mask]

accuracy = (pseudo_labels_confident == true_labels_confident).mean()
print(f"Pseudo-label accuracy on confident subset: {accuracy:.4f}")


Pseudo-label accuracy on confident subset: 0.9928


In [95]:
x_combined = np.vstack([x_labeled, x_unlabeled[confident_mask]])
y_combined = np.concatenate([y_labeled, pseudo_labels_confident])
print(x_combined.shape, y_combined.shape)
clf_self_trained = LogisticRegression(max_iter=1000, random_state=42)
clf_self_trained.fit(x_combined, y_combined)

y_pred_test = clf_self_trained.predict(x_test_scaled)

print(confusion_matrix(y_test_binary, y_pred_test))
print("Precision:", precision_score(y_test_binary, y_pred_test))
print("Recall:", recall_score(y_test_binary, y_pred_test))
print("F1 Score:", f1_score(y_test_binary, y_pred_test))





(110259, 122) (110259,)


/Users/sayoowo5/CS4360-Senior-Experience-Project/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/sayoowo5/CS4360-Senior-Experience-Project/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/sayoowo5/CS4360-Senior-Experience-Project/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/sayoowo5/CS4360-Senior-Experience-Project/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/sayoowo5/CS4360-Senior-Experience-Project/.venv/lib/python3.9/site-packages/sklearn/line

[[8965  746]
 [5116 7717]]
Precision: 0.9118515892709441
Recall: 0.6013402945531052
F1 Score: 0.7247370398196844


/Users/sayoowo5/CS4360-Senior-Experience-Project/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/sayoowo5/CS4360-Senior-Experience-Project/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/sayoowo5/CS4360-Senior-Experience-Project/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


In [96]:
clf_baseline = LogisticRegression(max_iter = 1000, random_state=42)
clf_baseline.fit(x_labeled, y_labeled)
y_pred_baseline = clf_baseline.predict(x_test_scaled)
print(confusion_matrix(y_test_binary, y_pred_baseline))
print("Precision:", precision_score(y_test_binary, y_pred_baseline))
print("Recall:", recall_score(y_test_binary, y_pred_baseline))
print("F1 Score:", f1_score(y_test_binary, y_pred_baseline))


[[8991  720]
 [5059 7774]]
Precision: 0.9152342830233106
Recall: 0.6057819683628146
F1 Score: 0.7290289304637314


/Users/sayoowo5/CS4360-Senior-Experience-Project/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/sayoowo5/CS4360-Senior-Experience-Project/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/sayoowo5/CS4360-Senior-Experience-Project/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/sayoowo5/CS4360-Senior-Experience-Project/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/sayoowo5/CS4360-Senior-Experience-Project/.venv/lib/python3.9/site-packages/sklearn/line